# Recipe Evaluation Pipeline
Notebook-first evaluation pipeline for recipe fine-tuning outputs.

In [ ]:
# EvalConfig (single place to configure columns, models, runtime and outputs)\n# This cell defines all runtime knobs so column remapping does not require code edits elsewhere.
from __future__ import annotations

import ast
import json
import math
import os
import re
import warnings
from collections import Counter
from datetime import datetime
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Set, Tuple

import numpy as np
import pandas as pd

try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    HF_AVAILABLE = True
except Exception:
    HF_AVAILABLE = False
    torch = None
    AutoModelForCausalLM = None
    AutoTokenizer = None

try:
    from bert_score import score as bertscore_score
    BERTSCORE_AVAILABLE = True
except Exception:
    BERTSCORE_AVAILABLE = False
    bertscore_score = None

RUN_ID = "20260322T165311Z_base_run"

# Central config used by loader, metrics, and artifact writer.
EVAL_CONFIG: Dict[str, Any] = {
    "input": {
        # Point this to your inference artifact. Supports CSV or Parquet.
        # Example: "../runs/2026-02-13_run001/predictions.parquet"
        "run_id": RUN_ID,
        "path": f"../runs/{RUN_ID}/predictions.parquet",
        "format": "parquet",  # "csv" or "parquet"
        # CSV-only options (ignored for parquet)
        "encoding": "utf-8",
        "engine": "python",
        "on_bad_lines": "warn",
    },
    "columns": {
        # Candidate names are tried in priority order.
        "recipe_id": ["recipe_id"],
        "title": ["title_normalized", "title"],
        "input": ["input_raw", "input"],
        "prediction": ["prediction","output", "model_output"],
        "reference": [ "reference","directions_normalized", "reference_directions",],
        # Pick ONE as primary ingredients column (names-only recommended for grounding metrics)
        "ingredients": ["ner_ingredients", "ingredients_names_only", "ingredients_normalized", "ingredients_bullets", "ingredients"],
    },
    "runtime": {
        "run_advanced": False,
        "batch_size": 4,
        "device": "auto",
        "max_rows": None,  # set to e.g., 100 for quick tests
        "strict_model_loading": False,
        "verbose": True,
    },
    "models": {
        "perplexity_model_name_or_path": "gpt2",
        "bertscore_model_type": None,
        "entailment_model": None,
    },
    "thresholds": {
        "ingredient_recall_pass": 0.50,
        "coherence_pass": 0.60,
        "completeness_pass": 4.0,
        "diversity_pass": 0.35,
    },
    "outputs": {
        # Writes artifacts into the same run folder by default.
        "dir": f"../runs/{RUN_ID}",
        "scored_prefix": "eval_scored",
        "summary_prefix": "eval_summary",
        "config_prefix": "eval_config",
    },
}

print("HF available:", HF_AVAILABLE, "| BERTScore available:", BERTSCORE_AVAILABLE)

In [ ]:
# Helpers: I/O, column resolution, text normalization and parsing\n# Keep parsing/normalization isolated so metric functions stay focused and testable.
STOPWORDS = {
    "a", "an", "and", "or", "the", "of", "for", "to", "with", "in", "on", "at", "by", "from", "into", "as",
    "is", "are", "be", "it", "this", "that", "these", "those", "then", "next", "after", "before", "until",
}

COMMON_UNITS = {
    "tsp", "teaspoon", "teaspoons", "tbsp", "tablespoon", "tablespoons", "cup", "cups", "oz", "ounce", "ounces",
    "lb", "lbs", "pound", "pounds", "g", "gram", "grams", "kg", "kilogram", "kilograms", "ml", "l", "liter",
    "liters", "can", "cans", "clove", "cloves", "pinch", "dash", "package", "packages", "pkg", "box", "boxes",
}

ACTION_VERBS = {
    "add", "bake", "boil", "broil", "chop", "combine", "cook", "cool", "dice", "drain", "fry", "garnish",
    "grill", "heat", "knead", "marinate", "mix", "peel", "pour", "preheat", "rest", "roast", "saute", "season",
    "serve", "simmer", "slice", "stir", "whisk",
}

ORDER_MARKERS = {"first", "then", "next", "after", "before", "meanwhile", "finally"}
FINISHING_WORDS = {"serve", "enjoy", "cool", "rest", "garnish", "store", "refrigerate"}

TIME_RE = re.compile(r"\b\d+\s*(min|mins|minute|minutes|hour|hours|hr|hrs)\b", re.IGNORECASE)
TEMP_RE = re.compile(r"\b\d+\s*(?:deg|degree|degrees)?\s*(?:f|c|fahrenheit|celsius)\b", re.IGNORECASE)
STEP_LINE_RE = re.compile(r"^\s*(?:\d+[\).]|step\s*\d+\b|[-*])", re.IGNORECASE)
TOKEN_RE = re.compile(r"[a-z0-9]+")
NUMERIC_RE = re.compile(r"^\d+(?:\.\d+)?(?:/\d+)?$")


def log(msg: str, config: Dict[str, Any]) -> None:
    if config["runtime"].get("verbose", True):
        print(msg)


def load_input_dataframe(config: Dict[str, Any]) -> pd.DataFrame:
    """Load the inference artifact produced by the Runpod notebook.

    Supports:
      - Parquet (recommended)
      - CSV (legacy)
    """
    cfg = config["input"]
    path = cfg.get("path") or cfg.get("csv_path")
    if path is None:
        raise KeyError('config["input"] must contain "path" (preferred) or "csv_path" (legacy).')

    fmt = (cfg.get("format") or ("csv" if str(path).lower().endswith(".csv") else "parquet")).lower()

    if fmt == "parquet":
        df = pd.read_parquet(path)
    elif fmt == "csv":
        df = pd.read_csv(
            path,
            encoding=cfg.get("encoding", "utf-8"),
            engine=cfg.get("engine", "python"),
            on_bad_lines=cfg.get("on_bad_lines", "warn"),
        )
    else:
        raise ValueError(f"Unsupported input format: {fmt}. Use 'csv' or 'parquet'.")

    max_rows = config["runtime"].get("max_rows")
    if isinstance(max_rows, int) and max_rows > 0:
        df = df.head(max_rows).copy()
    return df


def resolve_column(df: pd.DataFrame, candidates: Sequence[str], required: bool = True) -> Optional[str]:
    # Try candidate names in priority order (supports schema variation across files).
    for col in candidates:
        if col in df.columns:
            return col
    if required:
        raise KeyError(f"Required columns not found. Tried: {list(candidates)}")
    return None


def normalize_text(text: Any) -> str:
    if text is None or (isinstance(text, float) and math.isnan(text)):
        return ""
    if isinstance(text, list):
        text = "\n".join(str(x) for x in text)
    text = str(text)
    text = text.replace("\r", "\n")
    text = re.sub(r"[^\w\s\n\.,:%/-]", " ", text.lower())
    text = re.sub(r"\s+", " ", text).strip()
    return text


def split_lines(text: str) -> List[str]:
    raw = str(text).replace("\r", "\n").split("\n")
    return [line.strip() for line in raw if line.strip()]


def tokenize(text: str) -> List[str]:
    return TOKEN_RE.findall(normalize_text(text))


def safe_to_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and math.isnan(value):
        return ""
    if isinstance(value, list):
        return "\n".join(str(v) for v in value)
    return str(value)


def parse_ingredient_lines(value: Any) -> List[str]:
    # Accept list-like, Python-list string, or delimiter-separated formats.
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []

    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]

    text = str(value).strip()
    if not text:
        return []

    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except Exception:
            pass

    pieces = re.split(r"\n|;|\|", text)
    if len(pieces) <= 1:
        pieces = re.split(r",\s*(?=[0-9]|[a-zA-Z])", text)
    return [p.strip(" -\t\"'") for p in pieces if p.strip(" -\t\"'")]


def extract_ingredients_from_input(input_text: str) -> List[str]:
    # Fallback parser when dedicated ingredient column is missing/empty.
    text = safe_to_text(input_text)
    if not text.strip():
        return []

    lowered = text.lower()
    if "ingredients:" in lowered:
        block = text.split("Ingredients:", 1)[-1]
        for marker in ["Directions:", "Instructions:", "Method:"]:
            if marker.lower() in block.lower():
                block = re.split(marker, block, flags=re.IGNORECASE)[0]
                break
        return parse_ingredient_lines(block)
    return []


def canonical_ingredient(ingredient: str) -> str:
    # Normalize ingredient text to improve matching (remove qty/units/stopwords).
    tokens = tokenize(ingredient)
    cleaned = []
    for t in tokens:
        if t in STOPWORDS:
            continue
        if NUMERIC_RE.match(t):
            continue
        if t in COMMON_UNITS:
            continue
        cleaned.append(t)
    if cleaned and cleaned[-1].endswith("s") and len(cleaned[-1]) > 3:
        cleaned[-1] = cleaned[-1][:-1]
    return " ".join(cleaned).strip()


def ingredient_set(value: Any) -> Set[str]:
    lines = parse_ingredient_lines(value)
    out = set()
    for line in lines:
        c = canonical_ingredient(line)
        if c:
            out.add(c)
    return out


def ingredient_set_with_fallback(ingredient_col_value: Any, input_text: str) -> Set[str]:
    # Primary: explicit ingredient column. Secondary: parse from input prompt text.
    parsed = ingredient_set(ingredient_col_value)
    if parsed:
        return parsed
    fallback = extract_ingredients_from_input(input_text)
    out = set()
    for line in fallback:
        c = canonical_ingredient(line)
        if c:
            out.add(c)
    return out

In [ ]:
# Metrics: lexical metrics, completeness rubric, ingredient metrics, temporal and complexity metrics\n# Core metrics are deterministic and run for every sample.

def ngrams(tokens: List[str], n: int) -> List[Tuple[str, ...]]:
    if n <= 0 or len(tokens) < n:
        return []
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]


def sentence_bleu4(reference: str, prediction: str) -> float:
    # Lightweight BLEU-4 with add-1 smoothing for stable sentence-level scoring.
    ref = tokenize(reference)
    hyp = tokenize(prediction)
    if not ref or not hyp:
        return 0.0

    weights = [0.25, 0.25, 0.25, 0.25]
    precisions = []
    for n in range(1, 5):
        ref_counts = Counter(ngrams(ref, n))
        hyp_counts = Counter(ngrams(hyp, n))
        if not hyp_counts:
            precisions.append(0.0)
            continue
        clipped = 0
        total = 0
        for gram, count in hyp_counts.items():
            clipped += min(count, ref_counts.get(gram, 0))
            total += count
        # add-1 smoothing for sentence-level stability
        precisions.append((clipped + 1.0) / (total + 1.0))

    if min(precisions) <= 0:
        return 0.0

    ref_len = len(ref)
    hyp_len = len(hyp)
    bp = 1.0 if hyp_len > ref_len else math.exp(1 - (ref_len / max(hyp_len, 1)))
    score = bp * math.exp(sum(w * math.log(p) for w, p in zip(weights, precisions)))
    return float(max(min(score, 1.0), 0.0))


def lcs_length(a: List[str], b: List[str]) -> int:
    if not a or not b:
        return 0
    dp = [0] * (len(b) + 1)
    for i in range(1, len(a) + 1):
        prev = 0
        for j in range(1, len(b) + 1):
            cur = dp[j]
            if a[i - 1] == b[j - 1]:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j - 1])
            prev = cur
    return dp[-1]


def rouge_l_f1(reference: str, prediction: str) -> float:
    # ROUGE-L F1 computed from token-level longest common subsequence.
    ref = tokenize(reference)
    hyp = tokenize(prediction)
    if not ref or not hyp:
        return 0.0
    lcs = lcs_length(ref, hyp)
    precision = lcs / len(hyp)
    recall = lcs / len(ref)
    if precision + recall == 0:
        return 0.0
    return float((2 * precision * recall) / (precision + recall))


def repetition_ratio(text: str) -> float:
    tok = tokenize(text)
    if not tok:
        return 0.0
    return 1.0 - (len(set(tok)) / len(tok))


def distinct_n(text: str, n: int) -> float:
    tok = tokenize(text)
    grams = ngrams(tok, n)
    if not grams:
        return 0.0
    return len(set(grams)) / len(grams)


def diversity_score(text: str) -> float:
    # Combine lexical diversity (distinct-n) with repetition penalty.
    d1 = distinct_n(text, 1)
    d2 = distinct_n(text, 2)
    rep_penalty = 1.0 - repetition_ratio(text)
    return float(max(min(((d1 + d2) / 2.0) * rep_penalty, 1.0), 0.0))


def split_steps(text: str) -> List[str]:
    lines = split_lines(text)
    if not lines:
        return []
    numbered = [l for l in lines if STEP_LINE_RE.match(l)]
    if numbered:
        return numbered
    # fallback: sentence-level steps
    sents = [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]
    return sents


def ingredient_mentions_in_prediction(prediction: str, gold_ingredients: Set[str]) -> Set[str]:
    pred_norm = normalize_text(prediction)
    found = set()
    for ing in gold_ingredients:
        if ing and re.search(r"\b" + re.escape(ing) + r"\b", pred_norm):
            found.add(ing)
    return found


def ingredient_metrics(prediction: str, gold_ingredients: Set[str]) -> Dict[str, Any]:
    # Returns precision/recall/F1 + tracking diagnostics (missing/extra/hallucination).
    if not gold_ingredients:
        return {
            "ingredient_precision": 0.0,
            "ingredient_recall": 0.0,
            "ingredient_f1": 0.0,
            "ingredient_coverage_ratio": 0.0,
            "ingredient_missing_count": 0,
            "ingredient_extra_count": 0,
            "ingredient_hallucination_ratio": 0.0,
            "ingredient_missing_items": "",
            "ingredient_extra_items": "",
        }

    mentioned = ingredient_mentions_in_prediction(prediction, gold_ingredients)
    pred_tokens = set(tokenize(prediction))
    extras = {tok for tok in pred_tokens if tok in {g.split()[0] for g in gold_ingredients} and tok not in mentioned}

    tp = len(mentioned)
    fp = len(extras)
    fn = len(gold_ingredients - mentioned)

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 0.0 if (precision + recall == 0) else (2 * precision * recall / (precision + recall))
    halluc_ratio = fp / max(tp + fp, 1)

    return {
        "ingredient_precision": float(precision),
        "ingredient_recall": float(recall),
        "ingredient_f1": float(f1),
        "ingredient_coverage_ratio": float(recall),
        "ingredient_missing_count": int(fn),
        "ingredient_extra_count": int(fp),
        "ingredient_hallucination_ratio": float(halluc_ratio),
        "ingredient_missing_items": " | ".join(sorted(gold_ingredients - mentioned)),
        "ingredient_extra_items": " | ".join(sorted(extras)),
    }


def time_temp_spec_score(text: str) -> float:
    # Checks for explicit, usable time/temperature mentions.
    t = safe_to_text(text)
    has_time = bool(TIME_RE.search(t))
    has_temp = bool(TEMP_RE.search(t))
    numeric_with_unit = bool(re.search(r"\b\d+\s*(minutes?|hours?|f|c|fahrenheit|celsius)\b", t, flags=re.IGNORECASE))
    score = (1.0 if has_time else 0.0) + (1.0 if has_temp else 0.0) + (1.0 if numeric_with_unit else 0.0)
    return score / 3.0


def step_complexity_score(text: str) -> float:
    # Proxy complexity from step count, verb diversity, and multi-clause instructions.
    steps = split_steps(text)
    if not steps:
        return 0.0
    step_count_score = min(len(steps) / 8.0, 1.0)

    verbs_used = set()
    clause_scores = []
    for s in steps:
        tok = set(tokenize(s))
        verbs_used |= (tok & ACTION_VERBS)
        commas = s.count(",")
        connectors = len(re.findall(r"\b(and|then|while|until|after)\b", s, flags=re.IGNORECASE))
        clause_scores.append(min((commas + connectors) / 3.0, 1.0))

    verb_diversity_score = min(len(verbs_used) / 8.0, 1.0)
    clause_score = float(np.mean(clause_scores)) if clause_scores else 0.0

    return float((step_count_score + verb_diversity_score + clause_score) / 3.0)


def recipe_coherence_temporal_score(text: str) -> float:
    # Penalizes contradictory order (e.g., bake before preheat) and weak sequencing cues.
    raw = safe_to_text(text)
    norm = normalize_text(raw)
    if not norm:
        return 0.0

    score = 1.0
    preheat_pos = norm.find("preheat")
    bake_pos = min([p for p in [norm.find("bake"), norm.find("roast")] if p != -1], default=-1)
    if bake_pos != -1 and preheat_pos != -1 and bake_pos < preheat_pos:
        score -= 0.35

    if not any(marker in norm for marker in ORDER_MARKERS):
        score -= 0.15

    numbered = re.findall(r"(?:^|\n)\s*(\d+)[\).]", raw)
    if numbered:
        nums = [int(n) for n in numbered]
        if nums != sorted(nums):
            score -= 0.25

    if "serve" not in norm and "enjoy" not in norm:
        score -= 0.10

    return float(max(score, 0.0))


def step_entailment_score_core(reference: str, prediction: str) -> float:
    # Heuristic entailment proxy: overlap of meaningful action/content terms.
    ref_tokens = [t for t in tokenize(reference) if t not in STOPWORDS and len(t) > 2]
    pred_tokens = [t for t in tokenize(prediction) if t not in STOPWORDS and len(t) > 2]
    if not ref_tokens or not pred_tokens:
        return 0.0
    ref_set = set(ref_tokens)
    pred_set = set(pred_tokens)
    return float(len(ref_set & pred_set) / len(ref_set))


def recipe_completeness_0_7(
    # Expanded rubric score in [0, 7] from seven binary checks.
    prediction: str,
    ingredient_recall: float,
    step_count: int,
    time_temp_score: float,
) -> int:
    text = safe_to_text(prediction)
    norm = normalize_text(text)
    first_line = split_lines(text)[0] if split_lines(text) else ""

    checks = []
    checks.append(1 if len(first_line.split()) >= 2 else 0)  # has title/goal-like opener
    checks.append(1 if ingredient_recall >= 0.30 else 0)  # ingredient-grounded
    checks.append(1 if step_count >= 3 else 0)  # minimum steps
    checks.append(1 if any(m in norm for m in ORDER_MARKERS) else 0)  # sequencing markers
    checks.append(1 if len(set(tokenize(norm)) & ACTION_VERBS) >= 3 else 0)  # action verbs
    checks.append(1 if time_temp_score >= 0.50 else 0)  # time/temp
    checks.append(1 if any(w in norm for w in FINISHING_WORDS) else 0)  # finishing cue

    return int(sum(checks))

In [ ]:
# Model-backed metrics: Perplexity and BERTScore (with explicit status)\n# Failures are surfaced via status fields instead of silently dropping metrics.
class PerplexityScorer:
    # Small wrapper around HF causal LM to keep loading/scoring concerns encapsulated.
    def __init__(self, config: Dict[str, Any]):
        self.config = config
        self.model = None
        self.tokenizer = None
        self.status = "not_loaded"

    def load(self) -> None:
        if not HF_AVAILABLE:
            self.status = "transformers_or_torch_not_available"
            return

        model_name = self.config["models"]["perplexity_model_name_or_path"]
        device_cfg = self.config["runtime"].get("device", "auto")

        try:
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            if self.tokenizer.pad_token is None and self.tokenizer.eos_token is not None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
            self.model = AutoModelForCausalLM.from_pretrained(model_name)
            if device_cfg == "auto":
                device = "cuda" if torch.cuda.is_available() else "cpu"
            else:
                device = device_cfg
            self.model.to(device)
            self.model.eval()
            self.device = device
            self.status = f"loaded_on_{device}"
        except Exception as exc:
            self.status = f"load_failed: {exc}"
            self.model = None
            self.tokenizer = None

    def score_batch(self, texts: Sequence[str], max_length: int = 512) -> List[float]:
        if self.model is None or self.tokenizer is None:
            return [float("nan")] * len(texts)

        vals: List[float] = []
        for t in texts:
            t = safe_to_text(t)
            if not t.strip():
                vals.append(float("nan"))
                continue
            try:
                enc = self.tokenizer(
                    t,
                    return_tensors="pt",
                    truncation=True,
                    max_length=max_length,
                )
                enc = {k: v.to(self.device) for k, v in enc.items()}
                with torch.no_grad():
                    out = self.model(**enc, labels=enc["input_ids"])
                    loss = float(out.loss.item())
                vals.append(float(math.exp(min(loss, 20))))
            except Exception:
                vals.append(float("nan"))
        return vals


def compute_bertscore_batch(
    # Batch BERTScore with graceful fallback when package/model is unavailable.
    predictions: Sequence[str],
    references: Sequence[str],
    config: Dict[str, Any],
) -> Tuple[List[float], str]:
    if not BERTSCORE_AVAILABLE:
        return [float("nan")] * len(predictions), "bertscore_package_not_available"

    try:
        model_type = config["models"].get("bertscore_model_type")
        kwargs = {"lang": "en", "verbose": False}
        if model_type:
            kwargs["model_type"] = model_type
        _, _, f1 = bertscore_score(list(predictions), list(references), **kwargs)
        return [float(x) for x in f1.cpu().numpy().tolist()], "ok"
    except Exception as exc:
        return [float("nan")] * len(predictions), f"bertscore_failed: {exc}"

In [ ]:
# Advanced metric stubs and pipeline runner
# Advanced path is opt-in (un_advanced) and intentionally adapter-driven.

def llm_judge_recipe(
    # Adapter interface placeholder; intentionally raises if no adapter is provided.
    sample: Dict[str, Any],
    rubric: Dict[str, Any],
    adapter: Optional[Callable[[Dict[str, Any], Dict[str, Any]], Dict[str, Any]]] = None,
) -> Dict[str, Any]:
    """Strict stub interface for LLM-as-a-Judge.

    Contract:
      adapter(sample, rubric) -> {
          "overall_score": float,
          "verdict": str,
          "feedback": str,
          ...
      }
    """
    if adapter is None:
        raise NotImplementedError("LLM judge adapter is not configured.")
    return adapter(sample, rubric)


def advanced_step_entailment_model_score(
    reference: str,
    prediction: str,
    entailment_model: Optional[Any] = None,
) -> float:
    """Optional stronger entailment path. Implement by plugging an NLI model."""
    if entailment_model is None:
        raise NotImplementedError("Advanced entailment model is not configured.")
    raise NotImplementedError("Implement adapter-specific entailment scoring.")


def evaluate_dataframe(df: pd.DataFrame, config: Dict[str, Any]) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    # Main orchestrator: normalize input, compute metrics, aggregate summary, attach statuses.
    col_cfg = config["columns"]
    col_input = resolve_column(df, col_cfg["input"], required=True)
    col_pred = resolve_column(df, col_cfg["prediction"], required=True)
    col_ref = resolve_column(df, col_cfg["reference"], required=True)
    col_ing = resolve_column(df, col_cfg["ingredients"], required=False)

    work = df.copy()
    work["_input"] = work[col_input].apply(safe_to_text)
    work["_prediction"] = work[col_pred].apply(safe_to_text)
    work["_reference"] = work[col_ref].apply(safe_to_text)
    work["_ingredients_raw"] = work[col_ing] if col_ing else ""

    log(f"Rows: {len(work)} | input={col_input}, prediction={col_pred}, reference={col_ref}, ingredients={col_ing}", config)

    rows: List[Dict[str, Any]] = []
    for _, r in work.iterrows():
        input_text = r["_input"]
        pred = r["_prediction"]
        ref = r["_reference"]
        ing_set = ingredient_set_with_fallback(r["_ingredients_raw"], input_text)

        steps = split_steps(pred)
        ing_stats = ingredient_metrics(pred, ing_set)
        tts = time_temp_spec_score(pred)

        row_out: Dict[str, Any] = {
            "bleu4": sentence_bleu4(ref, pred),
            "rouge_l_f1": rouge_l_f1(ref, pred),
            "step_complexity_score": step_complexity_score(pred),
            "time_temp_spec_score": tts,
            "diversity_score": diversity_score(pred),
            "recipe_coherence_temporal_score": recipe_coherence_temporal_score(pred),
            "step_entailment_score": step_entailment_score_core(ref, pred),
            "num_steps": len(steps),
            "word_count": len(tokenize(pred)),
            "repetition_ratio": repetition_ratio(pred),
            "ingredients_source_count": len(ing_set),
        }
        row_out.update(ing_stats)
        row_out["recipe_completeness_0_7"] = recipe_completeness_0_7(
            pred,
            ingredient_recall=row_out["ingredient_recall"],
            step_count=row_out["num_steps"],
            time_temp_score=tts,
        )

        rows.append(row_out)

    metrics_df = pd.DataFrame(rows)

    # Perplexity
    ppl = PerplexityScorer(config)
    ppl.load()
    metrics_df["perplexity"] = ppl.score_batch(work["_prediction"].tolist())
    metrics_df["metric_status_perplexity"] = ppl.status

    # BERTScore
    bs_vals, bs_status = compute_bertscore_batch(work["_prediction"].tolist(), work["_reference"].tolist(), config)
    metrics_df["bertscore_f1"] = bs_vals
    metrics_df["metric_status_bertscore"] = bs_status

    if config["runtime"].get("strict_model_loading", False):
        if ppl.status.startswith("load_failed") or ppl.status == "transformers_or_torch_not_available":
            raise RuntimeError(f"Perplexity scorer unavailable: {ppl.status}")
        if bs_status != "ok":
            raise RuntimeError(f"BERTScore unavailable: {bs_status}")
            

    # Optional advanced metrics
    if config["runtime"].get("run_advanced", False):
        adv_errors = []
        entailment_model = config["models"].get("entailment_model")
        adv_vals = []
        for _, r in work.iterrows():
            try:
                adv_vals.append(advanced_step_entailment_model_score(r["_reference"], r["_prediction"], entailment_model))
            except NotImplementedError as exc:
                adv_vals.append(float("nan"))
                adv_errors.append(str(exc))

        metrics_df["step_entailment_score_advanced"] = adv_vals
        metrics_df["advanced_status_entailment"] = adv_errors[0] if adv_errors else "ok"

        # LLM judge stub only
        # LLM ERval in seperate notebook - this part is not used
        judge_scores = []
        judge_status = []
        for _, r in work.iterrows():
            sample = {
                "input": r["_input"],
                "prediction": r["_prediction"],
                "reference": r["_reference"],
            }
            rubric = {"name": "recipe_judge_v1"}
            try:
                out = llm_judge_recipe(sample, rubric, adapter=None)
                judge_scores.append(out.get("overall_score", float("nan")))
                judge_status.append("ok")
            except NotImplementedError as exc:
                judge_scores.append(float("nan"))
                judge_status.append(str(exc))

        metrics_df["llm_judge_overall_score"] = judge_scores
        metrics_df["llm_judge_status"] = judge_status

    scored = pd.concat([work.reset_index(drop=True), metrics_df.reset_index(drop=True)], axis=1)

    # Aggregate summary
    numeric_cols = scored.select_dtypes(include=[np.number]).columns.tolist()
    summary_rows = []
    for c in numeric_cols:
        summary_rows.append({
            "metric": c,
            "mean": float(np.nanmean(scored[c].values)) if len(scored) else float("nan"),
            "median": float(np.nanmedian(scored[c].values)) if len(scored) else float("nan"),
            "std": float(np.nanstd(scored[c].values)) if len(scored) else float("nan"),
        })

    thresholds = config["thresholds"]
    pass_block = {
        "pass_rate_ingredient_recall": float((scored["ingredient_recall"] >= thresholds["ingredient_recall_pass"]).mean()),
        "pass_rate_coherence": float((scored["recipe_coherence_temporal_score"] >= thresholds["coherence_pass"]).mean()),
        "pass_rate_completeness": float((scored["recipe_completeness_0_7"] >= thresholds["completeness_pass"]).mean()),
        "pass_rate_diversity": float((scored["diversity_score"] >= thresholds["diversity_pass"]).mean()),
    }

    summary_df = pd.DataFrame(summary_rows)
    for k, v in pass_block.items():
        summary_df = pd.concat([summary_df, pd.DataFrame([{"metric": k, "mean": v, "median": v, "std": 0.0}])], ignore_index=True)

    runtime_info = {
        "resolved_columns": {
            "input": col_input,
            "prediction": col_pred,
            "reference": col_ref,
            "ingredients": col_ing,
        },
        "metric_status": {
            "perplexity": ppl.status,
            "bertscore": bs_status,
        },
        "rows": len(scored),
    }

    return scored, summary_df, runtime_info


def save_artifacts(
    scored_df: pd.DataFrame,
    summary_df: pd.DataFrame,
    config: Dict[str, Any],
    runtime_info: Dict[str, Any],
) -> Dict[str, str]:

    # Persist row-level scores, summary stats, and resolved runtime/config metadata.
    out_dir = config["outputs"]["dir"]
    os.makedirs(out_dir, exist_ok=True)

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    scored_path = os.path.join(out_dir, f"{config['outputs']['scored_prefix']}_{ts}.csv")
    summary_path = os.path.join(out_dir, f"{config['outputs']['summary_prefix']}_{ts}.csv")
    config_path = os.path.join(out_dir, f"{config['outputs']['config_prefix']}_{ts}.json")

    scored_df.to_csv(scored_path, index=False)
    summary_df.to_csv(summary_path, index=False)

    payload = {
        "eval_config": config,
        "runtime_info": runtime_info,
    }
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

    return {
        "scored_path": scored_path,
        "summary_path": summary_path,
        "config_path": config_path,
    }

In [ ]:
# Run pipeline end-to-end\n# Execute these steps in order after updating EVAL_CONFIG.

# 1) Load data
_df = load_input_dataframe(EVAL_CONFIG)

# 2) Evaluate
scored_df, summary_df, runtime_info = evaluate_dataframe(_df, EVAL_CONFIG)

# 3) Save outputs
artifact_paths = save_artifacts(scored_df, summary_df, EVAL_CONFIG, runtime_info)

print("Runtime info:")
print(json.dumps(runtime_info, indent=2))
print("\nArtifacts:")
print(json.dumps(artifact_paths, indent=2))

# 4) Preview
display(scored_df.head(3))
display(summary_df.head(20))